# Planet Order Creation (Nepal Landslides)

Creates Planet orders for after images, and optionally before images.
Tracks all order attempts in Hugging Face at `raw_images/order_log.csv` to prevent duplicates.

In [1]:
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

# ----------------------------
# User configuration
# ----------------------------
INPUT_CSV = '/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv'
START_IDX = 721  # row-index start (inclusive)
END_IDX = 1500     # row-index end (exclusive); 0 means all rows
ORDER_BEFORE = True
REORDER_EXISTING = False

MAX_AOI_DEG = 0.1
CLOUD_MAX = 0.05              # <5%
PRE_DAYS = 180                # up to 6 months before
POST_DAYS = 60                # up to 2 months after (was 30 - too narrow to reliably find a
                               # cloud-free <5% scene, especially during/after monsoon cloud cover)
MAX_SCENES_PER_ORDER = 12     # allows multi-scene mosaic downstream

ORDERS_URL = 'https://api.planet.com/compute/ops/orders/v2'
DATA_URL = 'https://api.planet.com/data/v1'
ITEM_TYPE = 'PSScene'
BUNDLE_TYPE = 'analytic_sr_udm2'

HF_REPO_ID = 'sasudo2/landslides'
HF_REPO_TYPE = 'dataset'
HF_REVISION = 'main'
ORDER_LOG_PATH = 'raw_images/order_log.csv'
LOCAL_ORDER_LOG = '/kaggle/working/order_log.csv'

In [2]:
def clamp_aoi(min_lon, min_lat, max_lon, max_lat, max_deg=MAX_AOI_DEG):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= max_deg and lat_span <= max_deg:
        return float(min_lon), float(min_lat), float(max_lon), float(max_lat)
    cx = (min_lon + max_lon) / 2.0
    cy = (min_lat + max_lat) / 2.0
    half = max_deg / 2.0
    return float(cx - half), float(cy - half), float(cx + half), float(cy + half)

def polygon_from_bbox(min_lon, min_lat, max_lon, max_lat):
    return {
        'type': 'Polygon',
        'coordinates': [[[min_lon, min_lat], [max_lon, min_lat], [max_lon, max_lat], [min_lon, max_lat], [min_lon, min_lat]]],
    }

def make_planet_session(api_key):
    s = requests.Session()
    s.auth = (api_key, '')
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
    )
    s.mount('https://', HTTPAdapter(max_retries=retries))
    return s

def request_or_raise(resp):
    if not resp.ok:
        try:
            detail = resp.json()
        except Exception:
            detail = resp.text
        raise requests.HTTPError(f'{resp.status_code} {resp.reason} - {detail}', response=resp)

def search_scenes(session, aoi_geom, gte_ts, lte_ts):
    payload = {
        'item_types': [ITEM_TYPE],
        'filter': {
            'type': 'AndFilter',
            'config': [
                {'type': 'GeometryFilter', 'field_name': 'geometry', 'config': aoi_geom},
                {'type': 'DateRangeFilter', 'field_name': 'acquired', 'config': {'gte': gte_ts, 'lte': lte_ts}},
                {'type': 'RangeFilter', 'field_name': 'cloud_cover', 'config': {'lte': CLOUD_MAX}},
            ],
        },
    }
    r = session.post(f'{DATA_URL}/quick-search', json=payload, timeout=60)
    request_or_raise(r)
    return r.json().get('features', [])

def sort_features(features, incident_date):
    inc_dt = pd.Timestamp(incident_date)
    if inc_dt.tzinfo is not None:
        inc_dt = inc_dt.tz_localize(None)
    def key_fn(f):
        acquired = pd.to_datetime(f['properties']['acquired'])
        if acquired.tzinfo is not None:
            acquired = acquired.tz_localize(None)
        cloud = float(f['properties'].get('cloud_cover', 1.0) or 1.0)
        return (abs((acquired - inc_dt).total_seconds()), cloud)
    return sorted(features, key=key_fn)

def submit_order(session, order_name, item_ids, aoi_geom):
    payload = {
        'name': order_name,
        'products': [{'item_ids': item_ids, 'item_type': ITEM_TYPE, 'product_bundle': BUNDLE_TYPE}],
        'tools': [{'clip': {'aoi': aoi_geom}}],
    }
    r = session.post(ORDERS_URL, json=payload, timeout=60)
    request_or_raise(r)
    return r.json()

def list_orders(session):
    orders = []
    url = ORDERS_URL
    while url:
        r = session.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()
        orders.extend(data.get('orders', []))
        url = data.get('_links', {}).get('_next')
    return orders

In [3]:
# Auth + dataframe setup
secrets = UserSecretsClient()
planet_api_key = secrets.get_secret('Lokesh_planet')
hf_token = secrets.get_secret('huggingface_token')

planet = make_planet_session(planet_api_key)
probe = planet.get(ORDERS_URL, timeout=60)
request_or_raise(probe)
print('Planet auth OK')

hf_api = HfApi(token=hf_token)
hf_api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)
print('Hugging Face dataset ready')

df = pd.read_csv(INPUT_CSV)
if START_IDX == 0 and END_IDX == 0:
    df_sel = df.copy()
else:
    df_sel = df.iloc[START_IDX:END_IDX].copy()
df_sel['incident_on'] = pd.to_datetime(df_sel['incident_on'], dayfirst=True)
df_sel = df_sel.reset_index().rename(columns={'index': 'row_index'})
print(f'Incidents selected: {len(df_sel)}')

# Load prior order log from HF if present
try:
    downloaded = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        revision=HF_REVISION,
        filename=ORDER_LOG_PATH,
        token=hf_token,
    )
    prior_log = pd.read_csv(downloaded)
    print(f'Loaded existing order log rows: {len(prior_log)}')
except Exception:
    prior_log = pd.DataFrame()
    print('No existing order log found; starting fresh')

# Only dedupe on attempts that actually resulted in a live order.
# 'failed' attempts (transient errors, no scenes found yet, etc.) must be retried on the next run.
# 'skipped_no_after' (before order skipped because 'after' hadn't succeeded yet) must also be
# retried, since a later run's successful 'after' order should unblock the 'before' attempt.
NON_BLOCKING_STATES = {'failed', 'skipped_no_after'}

# The log only records the order's state *at submission time* - Planet always returns
# 'queued' right after a successful submit, and we never go back to check whether it later
# reached 'success' or instead expired/got purged from the account (which Planet does after
# a retention window). Trusting a logged 'queued'/'running' state forever would permanently
# block re-ordering incidents whose order silently expired before it was ever downloaded.
# So cross-check those against Planet's *live* order list and only treat them as blocking if
# the order still actually exists there.
STALE_CHECK_STATES = {'queued', 'running'}
try:
    live_order_names = {o.get('name', '') for o in list_orders(planet)}
    print(f'Live Planet orders found for expiry check: {len(live_order_names)}')
except Exception as e:
    live_order_names = None
    print(f'Warning: could not fetch live Planet orders for expiry check: {e}')

existing_keys = set()
expired_count = 0
if not prior_log.empty:
    for _, r in prior_log.iterrows():
        state = str(r.get('order_state', ''))
        if state in NON_BLOCKING_STATES:
            continue
        inc_id = int(r['incident_id'])
        order_type = str(r['order_type'])
        key = (inc_id, order_type)
        if state in STALE_CHECK_STATES and live_order_names is not None:
            order_name = f'incident_{inc_id}_planet_{order_type}'
            if order_name not in live_order_names:
                expired_count += 1
                continue
        existing_keys.add(key)
print(f'Existing keys after live-expiry check: {len(existing_keys)} (expired/purged, eligible for retry: {expired_count})')

Planet auth OK
Hugging Face dataset ready
Incidents selected: 779
No existing order log found; starting fresh


In [4]:
records = []
created = skipped = failed = skipped_no_after = 0

for _, row in df_sel.iterrows():
    inc_id = int(row['id'])
    incident_date = row['incident_on']
    min_lon, min_lat, max_lon, max_lat = clamp_aoi(row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    aoi = polygon_from_bbox(min_lon, min_lat, max_lon, max_lat)

    # Becomes False if this run's 'after' order search finds no scenes / fails outright -
    # without a valid after image there is nothing to difference against, so ordering
    # 'before' imagery in that case would just waste Planet quota.
    after_available = True

    for order_type in (['after'] + (['before'] if ORDER_BEFORE else [])):
        key = (inc_id, order_type)

        if order_type == 'before' and not after_available:
            skipped_no_after += 1
            print(f'[SKIP] incident_{inc_id}_planet_{order_type}: no after scenes found this run')
            records.append({
                'incident_id': inc_id, 'title': row['title'], 'incident_on': str(row['incident_on'].date()),
                'min_lon': min_lon, 'min_lat': min_lat, 'max_lon': max_lon, 'max_lat': max_lat,
                'row_index': int(row['row_index']), 'order_type': order_type, 'before_enabled': ORDER_BEFORE,
                'order_name': f'incident_{inc_id}_planet_{order_type}', 'order_id': '', 'order_state': 'skipped_no_after',
                'scene_count': 0, 'created_at': datetime.now(timezone.utc).isoformat(), 'error': 'no after scenes found this run'
            })
            continue

        if key in existing_keys and not REORDER_EXISTING:
            skipped += 1
            print(f'[SKIP] incident_{inc_id}_planet_{order_type}: already ordered in a previous run')
            records.append({
                'incident_id': inc_id, 'title': row['title'], 'incident_on': str(row['incident_on'].date()),
                'min_lon': min_lon, 'min_lat': min_lat, 'max_lon': max_lon, 'max_lat': max_lat,
                'row_index': int(row['row_index']), 'order_type': order_type, 'before_enabled': ORDER_BEFORE,
                'order_name': f'incident_{inc_id}_planet_{order_type}', 'order_id': '', 'order_state': 'skipped_existing',
                'scene_count': 0, 'created_at': datetime.now(timezone.utc).isoformat(), 'error': ''
            })
            continue

        if order_type == 'before':
            start = (incident_date - pd.DateOffset(days=PRE_DAYS)).strftime('%Y-%m-%dT00:00:00Z')
            end = (incident_date - pd.DateOffset(days=1)).strftime('%Y-%m-%dT23:59:59Z')
        else:
            start = (incident_date + pd.DateOffset(days=1)).strftime('%Y-%m-%dT00:00:00Z')
            end = (incident_date + pd.DateOffset(days=POST_DAYS)).strftime('%Y-%m-%dT23:59:59Z')

        order_name = f'incident_{inc_id}_planet_{order_type}'
        err = ''
        order_id = ''
        order_state = 'failed'
        scene_count = 0

        try:
            feats = search_scenes(planet, aoi, start, end)
            if len(feats) == 0:
                raise ValueError('No scenes found in date/cloud window')

            ranked = sort_features(feats, incident_date)
            chosen = ranked[:MAX_SCENES_PER_ORDER]
            item_ids = [f['id'] for f in chosen]
            scene_count = len(item_ids)

            order_json = submit_order(planet, order_name, item_ids, aoi)
            order_id = order_json.get('id', '')
            order_state = order_json.get('state', 'queued')
            created += 1
            print(f'[OK] {order_name}: {scene_count} scenes, state={order_state}')

            existing_keys.add(key)
            time.sleep(0.4)
        except Exception as e:
            failed += 1
            err = str(e)
            print(f'[FAIL] {order_name}: {err}')

        if order_type == 'after':
            after_available = order_state != 'failed'

        records.append({
            'incident_id': inc_id, 'title': row['title'], 'incident_on': str(row['incident_on'].date()),
            'min_lon': min_lon, 'min_lat': min_lat, 'max_lon': max_lon, 'max_lat': max_lat,
            'row_index': int(row['row_index']), 'order_type': order_type, 'before_enabled': ORDER_BEFORE,
            'order_name': order_name, 'order_id': order_id, 'order_state': order_state,
            'scene_count': scene_count, 'created_at': datetime.now(timezone.utc).isoformat(), 'error': err
        })

run_log = pd.DataFrame.from_records(records)
if prior_log.empty:
    merged = run_log
else:
    merged = pd.concat([prior_log, run_log], ignore_index=True)

merged.to_csv(LOCAL_ORDER_LOG, index=False)
hf_api.upload_file(
    path_or_fileobj=LOCAL_ORDER_LOG,
    path_in_repo=ORDER_LOG_PATH,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
)

print('---')
print(f'Created: {created}')
print(f'Skipped existing: {skipped}')
print(f'Skipped before (no after found): {skipped_no_after}')
print(f'Failed: {failed}')
print(f'Order log uploaded to: {ORDER_LOG_PATH}')

[OK] incident_75849_planet_after: 12 scenes, state=queued
[OK] incident_75849_planet_before: 12 scenes, state=queued
[OK] incident_75864_planet_after: 12 scenes, state=queued
[OK] incident_75864_planet_before: 12 scenes, state=queued
[OK] incident_75618_planet_after: 12 scenes, state=queued
[OK] incident_75618_planet_before: 12 scenes, state=queued
[OK] incident_75302_planet_after: 12 scenes, state=queued
[OK] incident_75302_planet_before: 12 scenes, state=queued
[OK] incident_75193_planet_after: 12 scenes, state=queued
[OK] incident_75193_planet_before: 12 scenes, state=queued
[OK] incident_75173_planet_after: 12 scenes, state=queued
[OK] incident_75173_planet_before: 12 scenes, state=queued
[OK] incident_75124_planet_after: 12 scenes, state=queued
[OK] incident_75124_planet_before: 12 scenes, state=queued
[OK] incident_75088_planet_after: 12 scenes, state=queued
[OK] incident_75088_planet_before: 12 scenes, state=queued
[OK] incident_75105_planet_after: 10 scenes, state=queued
[OK] i